# 📘 **End-to-End Pipeline: GPT‑4o Vision OCR → XLM‑R Embeddings → Azure ML → GPT‑4o Reasoning**
This notebook contains ALL steps you need as a beginner to build the full solution using the MultiFinBen-EnglishOCR dataset.

It includes:
1. Load dataset from Azure Storage (SAS URL)
2. GPT‑4o Vision OCR (with NEW corrected API format)
3. Budget‑safe processing loop
4. XLM‑R HuggingFace embeddings
5. Optional: Train fusion model
6. Prepare Azure Endpoint scoring script
7. GPT‑4o reasoning / summary generation


In [ ]:
!pip install openai pandas tqdm pyarrow scikit-learn joblib requests

In [ ]:
from openai import OpenAI
import pandas as pd
import time, os
from tqdm import tqdm
import requests, numpy as np, joblib

# 🔑 Insert your API keys here
OPENAI_API_KEY = 'YOUR_OPENAI_KEY'
HF_TOKEN = 'YOUR_HF_TOKEN'
client = OpenAI(api_key=OPENAI_API_KEY)

## 📥 Step 1 — Load MultiFinBen-EnglishOCR Dataset

In [ ]:
sas_url = 'https://<your-storage>.blob.core.windows.net/multifinben/raw/train-00000-of-00008.parquet?<sas>'
df = pd.read_parquet(sas_url)
df.head()

## 👁️ Step 2 — GPT‑4o Vision OCR (Correct API Format)

In [ ]:
def gpt4o_vision_ocr(base64_str):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{
            'role': 'user',
            'content': [
                { 'type': 'text', 'text': 'Extract ONLY the text from this image. Output plain text only.' },
                { 'type': 'image_url', 'image_url': { 'url': f'data:image/png;base64,{base64_str}' } }
            ]
        }]
    )
    return response.choices[0].message.content

## 💸 Step 3 — Budget‑Safe OCR Processing

In [ ]:
MAX_SPEND_USD = 8.0
COST_PER_IMAGE = 0.005
OUTPUT = 'ocr_budget_safe.pkl'

if 'ocr_text' not in df.columns:
    df['ocr_text'] = None

# Resume progress
if os.path.exists(OUTPUT):
    saved = pd.read_pickle(OUTPUT)
    df.update(saved)
    print('Resumed previous progress.')

total_spend = 0.0
processed = 0

for i in tqdm(range(len(df))):
    if isinstance(df.loc[i, 'ocr_text'], str) and len(df.loc[i, 'ocr_text']) > 0:
        continue
    if total_spend + COST_PER_IMAGE > MAX_SPEND_USD:
        print(f'STOP: Budget limit reached at ${total_spend:.2f}')
        break
    try:
        text = gpt4o_vision_ocr(df.loc[i, 'image'])
        df.loc[i, 'ocr_text'] = text
        total_spend += COST_PER_IMAGE
        processed += 1
        if processed % 20 == 0:
            df.to_pickle(OUTPUT)
            print('Progress saved.')
        time.sleep(0.3)
    except Exception as e:
        print('Error:', e)
        time.sleep(1)
        continue

df.to_pickle(OUTPUT)
print('OCR Done. Processed =', processed, '| Spend = $', total_spend)

## 🔡 Step 4 — Generate XLM‑R Embeddings

In [ ]:
EMB_URL = 'https://api-inference.huggingface.co/pipeline/feature-extraction/sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
headers = {'Authorization': f'Bearer {HF_TOKEN}'}

def embed_text(text):
    r = requests.post(EMB_URL, headers=headers, json={'inputs': text})
    return r.json()  # 768-dim vector


In [ ]:
df['embedding'] = df['ocr_text'].apply(embed_text)
df.to_pickle('processed_embeddings.pkl')
df.head()

## 🤖 Step 5 — Train a Fusion Model (Optional)

In [ ]:
from sklearn.linear_model import LogisticRegression

df_model = pd.read_pickle('processed_embeddings.pkl')
X = np.vstack(df_model['embedding'])
y = df_model['label']  # if available

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X, y)
joblib.dump(model, 'fusion_model.joblib')
model

## ☁️ Step 6 — Azure ML Real-Time Endpoint Scoring Script

In [ ]:
%%writefile score.py
import json, numpy as np, joblib, os

def init():
    global model
    path = os.environ['AZUREML_MODEL_DIR']
    model = joblib.load(os.path.join(path, 'fusion_model.joblib'))

def run(raw):
    d = json.loads(raw)
    emb = np.array(d['embedding'])
    score = float(model.predict_proba([emb])[0][1])
    return {'score': score}

## 📊 Step 7 — GPT‑4o Financial Reasoning & Summary

In [ ]:
def gpt4o_reason(text):
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role':'system','content':'You are a financial analyst.'},
            {'role':'user','content': text}
        ]
    )
    return response.choices[0].message.content

# Example:
gpt4o_reason(df['ocr_text'].iloc[0])